# Random Forests from Scratch: Bagging, Decorrelation, and Out-of-Bag Error

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/supervised/bagging_random_forest.ipynb)

Companion notebook to [Random Forests from Scratch](https://sesen.ai/blog/random-forests-from-scratch-bagging).

Object-oriented, NumPy only: a recursive `DecisionTree` node class wrapped in a `RandomForest` ensemble. The single parameter `max_features` separates plain bagging from a random forest.

> OOP structure inspired by lewtun's *hepml* 'Random Forest from Scratch' notebook (from the fast.ai ML course). Our version uses true bootstrap with replacement so out-of-bag scoring works.

## Setup

In [ ]:
# Uncomment on Colab
# !pip install scikit-learn matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

## The model: `DecisionTree` (recursive node) + `RandomForest` (ensemble)

`DecisionTree` builds itself on construction: `find_split` searches a random subset of `max_features` columns, then recurses into left/right child nodes. `RandomForest` bootstraps each tree and averages their votes.

In [ ]:
import numpy as np


class DecisionTree:
    def __init__(self, X, y, indices=None, max_features=None,
                 min_samples_leaf=5, max_depth=12, depth=0, rng=None):
        self.X, self.y = X, y
        self.indices = np.arange(len(y)) if indices is None else indices
        self.max_features = X.shape[1] if max_features is None else max_features
        self.min_samples_leaf = min_samples_leaf
        self.max_depth = max_depth
        self.depth = depth
        self.rng = np.random.default_rng() if rng is None else rng

        self.value = float(self.y[self.indices].mean())   # leaf prediction = P(class 1)
        self.score = float("inf")                          # lower is a better split
        self.feature_index = None
        self.split = None
        self.impurity_decrease = 0.0
        self.left = self.right = None
        if self.depth < self.max_depth and len(self.indices) >= 2 * self.min_samples_leaf:
            self._find_split()

    @property
    def is_leaf(self):
        return self.score == float("inf")

    def _find_split(self):
        n_feats = self.X.shape[1]
        # the random-feature step: only mtry columns are considered at this node
        candidates = self.rng.choice(n_feats, size=self.max_features, replace=False)
        for f in candidates:
            self._find_better_split(f)
        if self.is_leaf:
            return
        # impurity decrease at this node, for mean-decrease-in-impurity importance
        yn = self.y[self.indices]
        parent_var = (yn ** 2).sum() - yn.sum() ** 2 / len(yn)
        self.impurity_decrease = parent_var - self.score
        # partition and recurse
        x = self.X[self.indices, self.feature_index]
        left_mask = x <= self.split
        li, ri = self.indices[left_mask], self.indices[~left_mask]
        kw = dict(max_features=self.max_features, min_samples_leaf=self.min_samples_leaf,
                  max_depth=self.max_depth, depth=self.depth + 1, rng=self.rng)
        self.left = DecisionTree(self.X, self.y, li, **kw)
        self.right = DecisionTree(self.X, self.y, ri, **kw)

    def _find_better_split(self, f):
        x = self.X[self.indices, f]
        y = self.y[self.indices]
        order = np.argsort(x, kind="mergesort")
        xs, ys = x[order], y[order]
        n = len(ys)
        # running left/right sums to evaluate every threshold in one pass
        sum_y, sum_y2 = ys.sum(), (ys ** 2).sum()
        left_n = left_sum = left_sum2 = 0.0
        for i in range(n - 1):
            yi = ys[i]
            left_n += 1
            left_sum += yi
            left_sum2 += yi * yi
            right_n = n - left_n
            right_sum = sum_y - left_sum
            right_sum2 = sum_y2 - left_sum2
            if left_n < self.min_samples_leaf or right_n < self.min_samples_leaf:
                continue
            if xs[i] == xs[i + 1]:
                continue
            # weighted within-node variance after the split (the impurity to minimise)
            left_var = left_sum2 - left_sum ** 2 / left_n
            right_var = right_sum2 - right_sum ** 2 / right_n
            score = left_var + right_var
            if score < self.score:
                self.score = score
                self.feature_index = f
                self.split = (xs[i] + xs[i + 1]) / 2.0

    def predict(self, X):
        return np.array([self._predict_row(row) for row in X])

    def _predict_row(self, row):
        if self.is_leaf:
            return self.value
        node = self.left if row[self.feature_index] <= self.split else self.right
        return node._predict_row(row)

    def accumulate_importance(self, imp):
        if self.is_leaf:
            return
        imp[self.feature_index] += self.impurity_decrease
        self.left.accumulate_importance(imp)
        self.right.accumulate_importance(imp)


class RandomForest:
    def __init__(self, n_trees=100, sample_size=None, max_features="sqrt",
                 min_samples_leaf=5, max_depth=12, seed=0):
        self.n_trees = n_trees
        self.sample_size = sample_size
        self.max_features = max_features
        self.min_samples_leaf = min_samples_leaf
        self.max_depth = max_depth
        self.rng = np.random.default_rng(seed)

    def _resolve_mtry(self, n_features):
        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))
        if self.max_features in (None, "all"):
            return n_features            # <- this makes the forest plain bagging
        return min(self.max_features, n_features)

    def fit(self, X, y):
        self.X, self.y = X, y
        n = len(y)
        m = n if self.sample_size is None else self.sample_size
        mtry = self._resolve_mtry(X.shape[1])
        self.trees, self.oob_indices = [], []
        for _ in range(self.n_trees):
            boot = self.rng.integers(0, n, size=m)         # bootstrap: with replacement
            oob = np.setdiff1d(np.arange(n), np.unique(boot))
            tree = DecisionTree(X, y, indices=boot, max_features=mtry,
                                min_samples_leaf=self.min_samples_leaf,
                                max_depth=self.max_depth,
                                rng=np.random.default_rng(self.rng.integers(1 << 30)))
            self.trees.append(tree)
            self.oob_indices.append(oob)
        return self

    def predict_proba(self, X):
        return np.mean([t.predict(X) for t in self.trees], axis=0)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

    def tree_predictions(self, X):
        """Per-tree prediction matrix (n_trees, n_samples) for correlation analysis."""
        return np.array([t.predict(X) for t in self.trees])

    def staged_proba(self, X):
        """Cumulative ensemble probability using the first 1, 2, ... trees."""
        P = self.tree_predictions(X)
        return np.cumsum(P, axis=0) / np.arange(1, len(self.trees) + 1)[:, None]

    def feature_importances_(self):
        imp = np.zeros(self.X.shape[1])
        for t in self.trees:
            t.accumulate_importance(imp)
        total = imp.sum()
        return imp / total if total > 0 else imp

    def oob_score(self):
        """Out-of-bag accuracy: each row scored only by trees that did not train on it."""
        n = len(self.y)
        votes = np.zeros(n)
        counts = np.zeros(n)
        for tree, oob in zip(self.trees, self.oob_indices):
            if len(oob) == 0:
                continue
            votes[oob] += tree.predict(self.X[oob])
            counts[oob] += 1
        seen = counts > 0
        pred = (votes[seen] / counts[seen] >= 0.5).astype(int)
        return float((pred == self.y[seen]).mean())

## A single tree overfits; the forest smooths

On two-moons, watch the boundary stabilise as trees are averaged.

In [ ]:
from sklearn.datasets import make_moons
from matplotlib.colors import ListedColormap
Xm, ym = make_moons(300, noise=0.30, random_state=0); ym = ym.astype(float)
G = 60; xx, yy = np.meshgrid(np.linspace(Xm[:,0].min()-.5, Xm[:,0].max()+.5, G),
                             np.linspace(Xm[:,1].min()-.5, Xm[:,1].max()+.5, G))
grid = np.c_[xx.ravel(), yy.ravel()]
forest = RandomForest(100, max_features='sqrt', min_samples_leaf=1, max_depth=14, seed=0).fit(Xm, ym)
staged = forest.staged_proba(grid)
cm = ListedColormap(['#dc2626','#2563eb'])
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, k in zip(ax, [1, 5, 25, 100]):
    p = staged[k-1].reshape(xx.shape)
    a.contourf(xx, yy, p, levels=20, cmap='RdBu', alpha=.6, vmin=0, vmax=1)
    a.contour(xx, yy, p, [0.5], colors='k'); a.scatter(Xm[:,0], Xm[:,1], c=ym, cmap=cm, s=20, edgecolor='w')
    a.set_title(f'{k} tree(s)'); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## One knob: bagging vs random forest

`max_features='all'` is bagging; `max_features='sqrt'` is a random forest. We measure inter-tree correlation and accuracy on a 40-feature problem.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
X, y = make_classification(1000, n_features=40, n_informative=8, n_redundant=6,
                           n_clusters_per_class=2, class_sep=0.9, flip_y=0.05, random_state=2)
y = y.astype(float)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

def mean_corr(forest, X):
    P = forest.tree_predictions(X); P = P - P.mean(1, keepdims=True)
    nrm = np.linalg.norm(P, axis=1, keepdims=True); nrm[nrm==0] = 1e-9
    C = (P @ P.T) / (nrm @ nrm.T)
    return C[np.triu_indices(len(forest.trees), 1)].mean()
acc = lambda p, t: (p == t).mean()

bag = RandomForest(100, max_features='all',  min_samples_leaf=1, max_depth=16, seed=0).fit(Xtr, ytr)
rf  = RandomForest(100, max_features='sqrt', min_samples_leaf=1, max_depth=16, seed=0).fit(Xtr, ytr)
print(f'bagging: corr {mean_corr(bag, Xte):.3f}  acc {acc(bag.predict(Xte), yte):.3f}')
print(f'forest : corr {mean_corr(rf, Xte):.3f}  acc {acc(rf.predict(Xte), yte):.3f}')

## Out-of-bag error: a free validation set

Each tree omits ~37% of rows (the `$(1-1/n)^n \to 1/e$` left out). Scoring each row with only the trees that never saw it gives an unbiased error estimate.

In [ ]:
print(f'RF OOB accuracy  {rf.oob_score():.3f}')
print(f'RF test accuracy {acc(rf.predict(Xte), yte):.3f}')

## Feature importance (mean decrease in impurity)

In [ ]:
from sklearn.datasets import load_breast_cancer
Xb, yb = load_breast_cancer(return_X_y=True); names = load_breast_cancer().feature_names; yb = yb.astype(float)
rfb = RandomForest(150, max_features='sqrt', min_samples_leaf=3, max_depth=14, seed=0).fit(Xb, yb)
imp = rfb.feature_importances_(); order = np.argsort(imp)[::-1][:10][::-1]
plt.barh(range(len(order)), imp[order], color='#059669')
plt.yticks(range(len(order)), [names[i] for i in order]); plt.xlabel('mean decrease in impurity'); plt.show()

## Validate against scikit-learn

In [ ]:
from sklearn.ensemble import RandomForestClassifier
Xa, Xc, ya, yc = train_test_split(Xb, yb, test_size=0.3, random_state=0, stratify=yb)
ours = RandomForest(100, max_features='sqrt', min_samples_leaf=3, seed=0).fit(Xa, ya)
sk = RandomForestClassifier(100, max_features='sqrt', min_samples_leaf=3, random_state=0).fit(Xa, ya)
print(f'from scratch {acc(ours.predict(Xc), yc):.3f} | sklearn {acc(sk.predict(Xc).astype(int), yc):.3f}')

## What to remember
- **Bagging** averages trees on bootstrap samples to cut **variance**; bias barely moves.
- The variance of correlated trees has a floor `$\rho\sigma^2$`; **feature subsampling** lowers `$\rho$`, which is what makes it a *random forest*.
- **Out-of-bag** rows (~37%) give a free validation estimate.
- You cannot overfit by adding trees; error converges to a floor.

## Exercises
1. **Sweep max_features** from 1 to p. Plot inter-tree correlation and test accuracy against it. Where is the sweet spot?
2. **Regression forest.** Adapt the leaf to predict the mean and average real-valued targets; test on a regression dataset.
3. **OOB vs CV.** Compare the OOB estimate to 5-fold cross-validation accuracy. How close are they?
4. **Permutation importance.** Shuffle each feature in the test set and measure the accuracy drop; compare to MDI.
5. **Multiclass.** Extend the impurity to Gini over more than two classes and test on a 3-class problem.